# Tech Challenge Fase 3: predição da alfabetização

Roteiro do projeto, na ordem: dados da Gold (Fase 2) → análise exploratória → pipeline de ML → validação → interpretação → perguntas de negócio.

Todo o código está em `src/`; o notebook chama as etapas e mostra os resultados.

## 0. Ambiente

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/daianeaxr/tech-challenge-fase3.git
    %cd tech-challenge-fase3
    !pip install -q -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, os.getcwd())
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 50)

from src import config
img = lambda nome: display(Image(str(config.IMAGES / f"{nome}.png")))

## 1. Extração

Fontes possíveis:
- `bigquery`: mesmas tabelas da Fase 2 direto na Base dos Dados (precisa do JSON da service account do GCP);
- `athena`: lê Silver/Gold da Fase 2 na AWS (precisa da infra da Fase 2 de pé).

Se `data/raw/` já estiver preenchido, pule esta célula.

In [ ]:
FONTE = "bigquery"   # None para pular

if IN_COLAB and FONTE == "bigquery":
    from google.colab import files
    enviado = files.upload()
    os.environ["GCP_SERVICE_ACCOUNT_JSON"] = os.path.abspath(next(iter(enviado)))
else:
    os.environ.setdefault("GCP_SERVICE_ACCOUNT_JSON", "")  # caminho local do JSON

from src.preprocessing import extract
if FONTE:
    extract.main(FONTE)

## 2. Bases analíticas

In [ ]:
from src.preprocessing import build_dataset
build_dataset.main()

aluno = pd.read_parquet(config.DATA_PROCESSED / "base_aluno.parquet")
mun = pd.read_parquet(config.DATA_PROCESSED / "base_municipio.parquet")
print(aluno.shape, mun.shape)
aluno.head()

In [ ]:
# tipos e faltantes da base do aluno
pd.DataFrame({"tipo": aluno.dtypes.astype(str), "% faltante": (aluno.isna().mean() * 100).round(1)})

## 3. Análise exploratória

Objetivo: entender distribuições e relações antes de modelar e usar isso para escolher as features.

In [ ]:
from src.visualization import eda
resumo = eda.main()
resumo

In [ ]:
for nome in ["eda_01_regiao", "eda_03_rede", "eda_02_uf"]:
    img(nome)

**Hipótese 1:** o resultado anterior do município antecipa o resultado do aluno. Taxa de 2024 por quintil do resultado de 2023:

In [ ]:
img("eda_04_historico_vs_aluno")

**Hipótese 2:** renda e estrutura econômica do município estão associadas ao resultado.

In [ ]:
img("eda_05_pib_vs_taxa")
img("eda_06_correlacao")

**Hipótese 3:** parte relevante dos municípios fica abaixo da própria meta, o que justifica um modelo de risco por município.

In [ ]:
img("eda_07_gap_meta")
mun[mun.ano_ref == 2024].nao_atingiu_meta.value_counts(normalize=True)

**Como a EDA guiou a modelagem**

- histórico do município (2023) entra como feature principal, na mesma rede do aluno e na rede pública;
- PIB per capita é muito assimétrico, por isso recebe `log1p` dentro do pipeline;
- há municípios sem resultado em 2023 e sem PIB, por isso imputação pela mediana com indicador de faltante;
- UF e região têm efeito claro e entram com one-hot;
- alunos da mesma escola se parecem, então treino e teste são separados por escola.

## 4. Pipeline de Machine Learning

In [ ]:
from src.modeling.pipeline import FEATURES_ALUNO, criar_pipeline
criar_pipeline(FEATURES_ALUNO, "aluno", "gradient_boosting")

Etapas dentro do `Pipeline`:
1. `FeatureEngineering`: cria `gap_meta_historico` (meta 2024 menos resultado 2023);
2. `ColumnTransformer`: imputação (mediana + indicador / moda), escala, `log1p` no PIB, one-hot;
3. classificador.

Como tudo está num único objeto, cada dobra da validação cruzada reajusta imputação e escala só com o próprio treino.

## 5. Treino, otimização e validação

In [ ]:
from src.modeling import train
rel_aluno = train.main()
pd.DataFrame(rel_aluno["modelos"]).T

In [ ]:
print("Modelo escolhido:", rel_aluno["modelo_escolhido"])
pd.DataFrame({"teste": rel_aluno["teste"], "baseline": rel_aluno["baseline"]}).drop("matriz_confusao")

In [ ]:
img("aluno_comparacao_modelos")
img("aluno_avaliacao")

### Conferência de vazamento

In [ ]:
prev = pd.read_parquet(config.DATA_PROCESSED / "aluno_previsoes_teste.parquet")
escolas_teste = set(prev.id_escola)
escolas_treino = set(aluno.id_escola) - escolas_teste
print("escolas em treino e teste ao mesmo tempo:", len(escolas_treino & escolas_teste))
print("proficiencia na base de modelagem?", "proficiencia" in aluno.columns)
print("anos na base de modelagem:", aluno.ano.unique())

## 6. Interpretabilidade (SHAP e Permutation Importance)

In [ ]:
img("aluno_shap_importancia")
img("aluno_shap_beeswarm")
img("aluno_permutation_importance")
pd.read_csv(config.REPORTS / "aluno_importancia.csv").head(10)

## 7. Perguntas de negócio: municípios

In [ ]:
from src.modeling import municipios
rel_mun = municipios.main()
print("Modelo:", rel_mun["modelo_escolhido"], "| AUC teste:", rel_mun["teste"]["roc_auc"])
rel_mun["projecao"]

**Quais municípios apresentam maior risco de não atingir a meta de 2025?**

In [ ]:
ranking = pd.read_csv(config.REPORTS / "ranking_risco_meta_2025.csv", dtype={"id_municipio": str})
img("municipio_risco_por_uf")
ranking.head(20)

**Quais regiões possuem padrões semelhantes?**

In [ ]:
img("municipio_clusters")
pd.read_csv(config.REPORTS / "clusters_perfil.csv")

**Quais variáveis mais influenciam o risco do município?**

In [ ]:
img("municipio_shap_importancia")
pd.read_csv(config.REPORTS / "municipio_importancia.csv").head(10)

## 8. README e apresentação

In [ ]:
from src.visualization import relatorio, slides
relatorio.main()   # atualiza a seção de resultados do README
slides.main()      # gera reports/apresentacao_fase3.pptx